# Experiments

NOTE: this file may take some time to run! 

In [2]:
import numpy as np
import prh as prh
import matplotlib.pyplot as plt
import plot as pltter
import generate_data as gen
import oat_python as oat

In [19]:
F = [] # full complex simplices counts
G = [] # subcomplex simplices counts
T = [] # execution time

dd = np.linspace(4, 1, 40)
for d in dd: 
    
    # generate data 
    X = gen.randomized_grid_sample_points(radius=2.2, dx=d, dy=d, stack=True) 

    # select length of subcloud indices array for each cloud x in X
    NN = X.shape[0]-1
    Ns = np.linspace(1, NN, 4) 
    Ns = np.round(Ns, decimals=0).astype(int)
    for n in Ns:
        if n == 0: n=1

        # randomly generate the subcloud indices for each cloud x in X
        subcloud_indices = np.random.choice(
            np.array(range(X.shape[0])), 
            size=n, 
            replace=False
        )

        # create PRH object 
        epsilon = oat.dissimilarity.enclosing_radius_for_points(X)
        alg_manager = prh.prh(
            X, 
            np.column_stack([subcloud_indices, np.full(n, 0.0)]), 
            1, 
            epsilon, 
            benchmarking=True
        )

        # count number of simplices in full complex and subcomplex 
        F.append(alg_manager.boundary_matrix.shape[0])
        G.append(alg_manager.boundary_matrix.bifiltration.num_subcomplex_simplices)

        # run PRH and time it
        homology_data, time_data = alg_manager.main_algorithm()
        T.append(time_data)
    
# plot the collected data
fig, ax = plt.subplots()
scatter = ax.scatter(
    F,
    G,
    c=T,
    cmap="viridis"
)
mF = np.max(F)
mG = np.max(G)
m = max(mF, mG)
ax.set_xlim(0, m)
ax.set_ylim(0, m)
ax.set_aspect("equal")
ax.set_xlabel("# of Simplices")
ax.set_ylabel("# of Simplices (in subcomplex)")
fig.colorbar(scatter, ax=ax, label="runtime (seconds)")
plt.show()


KeyboardInterrupt: 